In [0]:
# Listando os arquivos disponíveis (formato Parquet, que é o ideal para otimização)
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/"))



In [0]:
# Definindo o caminho do dataset gigante (Yellow Taxi Trips)
path_taxi = "/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/"

# Lendo os dados 
df_taxi = spark.read.format("delta").load(path_taxi)

# Vamos ver o tamanho do "problema"
print(f"Total de registros: {df_taxi.count():,}")
# 1,611,611,035 registros (mais de 1bi)

# Mostrando o esquema do dataset
df_taxi.printSchema()


In [0]:
# Mostra um exemplo dos dados
display(df_taxi.limit(50))

In [0]:
from pyspark.sql import functions as F
import time

# Teste 1: sem otimização
start_time = time.time()
res1 = df_taxi.select("*").filter(F.col("fare_amount") > 10).agg(F.avg("fare_amount"))
res1.show()
print(f"Tempo sem otimização: {time.time() - start_time:.2f}s")

# teste 2: com filtro precoce e seleção de colunas (Catalyst amigável)
start_time = time.time()
res2 = df_taxi.select("fare_amount").filter(F.col("fare_amount") > 10).agg(F.avg("fare_amount"))
res2.show()
print(f"Tempo com otimização: {time.time() - start_time:.2f}s")




In [0]:
res2.explain()

In [0]:
start_time = time.time()

df_filtered = df_taxi.filter(F.year("pickup_datetime") == 2015) \
                     .select("vendor_id", "fare_amount", "trip_distance")

print(f"Total de registros filtrados: {df_filtered.count():,}")
print(f"Tempo de filtragem: {time.time() - start_time:.2f}s")

df_filtered.explain()


In [0]:
# filtrando apenas as corridas de janeiro de 2015
df_jan_2015 = df_taxi.filter((F.col("pickup_datetime") >= "2015-01-01") &
                             (F.col("pickup_datetime") < "2015-02-01"))
                     
print(f"Registros em Janeiro/2015: {df_jan_2015.count():,}")      
print(f"Tempo com Predicate Pushdown: {time.time() - start_time:.2f}s")               

df_jan_2015.explain()
                     

In [0]:
# mostra 50 registros de exemplo de df_jan_2015
display(df_jan_2015.limit(500))


In [0]:
# mostra todos os valores de payment_type dentro de df_jan_2015
display(df_jan_2015.select("payment_type").distinct().orderBy("payment_type"))

In [0]:
# agora usando Tungsten
start = time.time()

# agregação pesada: o Tungsten vai usar whole-Stage Code Generation aqui
df_stats = df_jan_2015.groupBy("payment_type").agg(
    F.avg("tip_amount").alias("media_gorjeta"),
    F.max("fare_amount").alias("tarifa_maxima"),
    F.count("*").alias("total_corridas")
).orderBy("payment_type")

df_stats.show()
print(f"Tempo com Tungsten: {time.time() - start:.2f}s")
#

In [0]:
catalog = "nyc_transit"
schema = "yellow_taxi"

In [0]:
tabela_destino = "nyc_transit.yellow_taxi.trip_optimized"

# criando a tabela a partir dos dados lidos
# vamos usar apenas os dados de 2015
df_jan_2015.write.mode("overwrite").saveAsTable(tabela_destino)



In [0]:
# mostra a tabela criada
display(spark.sql(f"SELECT * FROM {tabela_destino} LIMIT 50"))

In [0]:
df_taxi_columns = spark.table(tabela_destino)

print(df_taxi_columns.columns)

In [0]:
import time
from pyspark.sql import functions as F

# Definindo uma área de busca (aproximadamente Midtown Manhattan)
lat_min, lat_max = 40.75, 40.85
lon_min, lon_max = -73.95, -73.85

start_time = time.time()

# busca cega
count_antes = (spark.table(tabela_destino)
               .filter(F.col("pickup_latitude").between(lat_min, lat_max))
               .filter(F.col("pickup_longitude").between(lon_min, lon_max))
               .count())

tempo_antes = time.time() - start_time

print(f"Total de registros: {count_antes:,}")
print(f"Tempo de busca cega: {tempo_antes:.2f}s")


In [0]:
%sql

OPTIMIZE nyc_transit.yellow_taxi.trip_optimized ZORDER BY (pickup_latitude, pickup_longitude)

In [0]:
# conta a quantidade de registros da tabela destino
count_depois = spark.table(tabela_destino).count()

print(f"Total de registros: {count_depois:,}")
print(f"Tempo de busca cega: {tempo_antes:.2f}s")


In [0]:
import time
from pyspark.sql import functions as F

# Área de busca para o Desembarque (ex: Times Square Area)
d_lat_min, d_lat_max = 40.757, 40.759
d_lon_min, d_lon_max = -73.986, -73.984

start = time.time()

# faz a busca por dropoff
count_dropoff_antes = (spark.table(tabela_destino)
                      .filter((F.col("dropoff_latitude").between(d_lat_min, d_lat_max)) & 
                              (F.col("dropoff_longitude").between(d_lon_min, d_lon_max)))
                      .count())

print(f"✅ Busca por Dropoff (Antes): {count_dropoff_antes:,} registros")
print(f"Tempo: {time.time() - start:.2f}s")

In [0]:
%sql
-- Reorganizando a tabela para performance em Pickup e Dropoff simultaneamente
OPTIMIZE nyc_transit.yellow_taxi.trip_optimized
ZORDER BY (pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude)

In [0]:
start = time.time()

# faz a busca por dropoff
count_dropoff_depois = (spark.table(tabela_destino)
                      .filter((F.col("dropoff_latitude").between(d_lat_min, d_lat_max)) & 
                              (F.col("dropoff_longitude").between(d_lon_min, d_lon_max)))
                      .count())

print(f"✅ Busca por Dropoff (Após Z-Order 4D): {count_dropoff_depois:,} registros")
print(f"Tempo: {time.time() - start:.2f}s")

In [0]:
# buscando o caminho físico da tabela nos metadados do Unit Catalog
table_info = spark.sql(f"DESCRIBE DETAIL nyc_transit.yellow_taxi.trip_optimized")

# mostra o conteudo de table_info
display(table_info)

#path_tabela = table_info.select("location").collect()[0][0]
# mostrando o caminho da tabela
#print(f"O caminho físico da sua tabela é: {path_tabela}")


In [0]:
%sql
DESCRIBE EXTENDED nyc_transit.yellow_taxi.trip_optimized

In [0]:
# Tente listar o diretório padrão de tabelas gerenciadas
try:
    caminho_default = "dbfs:/user/hive/warehouse/nyc_transit.db/yellow_taxi.db/nyc_optimized"
    display(dbutils.fs.ls(caminho_default))
except:
    print("Caminho não encontrado no local padrão. Verifique o DESCRIBE EXTENDED.")

In [0]:
%sql
SELECT *
-- table_catalog, table_schema, table_name, table_type, storage_sub_directory, storage_path
FROM nyc_transit.information_schema.tables
WHERE table_schema = 'yellow_taxi'
  AND table_name = 'trip_optimized';

  --tables/6c68ec48-0a54-4376-bc4c-05d5bd1039e6
  --tables/6c68ec48-0a54-4376-bc4c-05d5bd1039e6
                      

In [0]:
%sql
SELECT *
FROM system.information_schema.tables
WHERE table_catalog = 'nyc_transit'
  AND table_schema = 'yellow_taxi'
  AND table_name = 'trip_optimized';

In [0]:
%sql 
SHOW TBLPROPERTIES nyc_transit.yellow_taxi.trip_optimized;

In [0]:
%sql

VACUUM nyc_transit.yellow_taxi.trip_optimized RETAIN 0 HOURS

In [0]:
# Esta configuração permite ignorar o limite de 7 dias
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [0]:
%sql
--visualizar o time travel da tabela
DESCRIBE HISTORY nyc_transit.yellow_taxi.trip_optimized


In [0]:
%sql
VACUUM nyc_transit.yellow_taxi.trip_optimized

In [0]:
%sql

ALTER TABLE nyc_transit.yellow_taxi.trip_optimized
RENAME TO nyc_transit.yellow_taxi.trip_optimized;

In [0]:
%sql
DESCRIBE HISTORY nyc_transit.yellow_taxi.trip_optimized;

In [0]:
%sql

DESCRIBE EXTENDED nyc_transit.yellow_taxi.trip_optimized;

In [0]:
%sql
SELECT * FROM nyc_transit.yellow_taxi.trip_optimized
WHERE tip_amount > 0 AND total_amount < 500;

In [0]:
from pyspark.sql import functions as F

precisao = 3

top_rotas_gorjetas = (spark.table(tabela_destino)
                      .filter("tip_amount > 0 AND total_amount < 500")
                      .withColumn("lat_origem", F.round("pickup_latitude", precisao))
                      .withColumn("lon_origem", F.round("pickup_longitude", precisao))
                     # .withColumn("lat_destino", F.round("dropoff_latitude", precisao))
                     # .withColumn("lon_destino", F.round("dropoff_longitude", precisao))
#                      .groupBy("lat_origem", "lon_origem", "lat_destino", "lon_destino")
                      .groupBy("lat_origem", "lon_origem")
                      .agg(
                          F.count("*").alias("total_viagens"),
                          F.round(F.avg("tip_amount"), 2).alias("media_gorjeta"),
                          F.round(F.sum("tip_amount"), 2).alias("faturamento_gorjeta")
                      )
                      .filter("total_viagens > 50")
                      .orderBy(F.desc("media_gorjeta"))
                      .limit(10)
)

display(top_rotas_gorjetas)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

# filtrando as corridas de 2016 em diante
df_2016_plus = df_taxi.filter((F.col("pickup_datetime") >= "2016-01-01"))
                     
print(f"Registros de 2016 em diante: {df_2016_plus.count():,}")      

df_2016_plus.explain()

In [0]:
from pyspark.sql import functions as F

# filtrando as corridas de 2015 em diante
df_2015_plus = df_taxi.filter((F.col("pickup_datetime") >= "2015-01-01"))
                     
print(f"Registros de 2015 em diante: {df_2015_plus.count():,}")      

df_2015_plus.explain()

In [0]:
from pyspark.sql import functions as F

# filtrando as corridas de 2000 em diante
df_2009_2019 = df_taxi.filter((F.col("pickup_datetime") >= "2009-01-01")) 
#                              (F.col("pickup_datetime") <= "2026-12-31"))
                     
print(f"Registros de 2009 a 2019 em diante: {df_2009_2019.count():,}")      

# agrupa ano para saber a quantidade anual, ordenado por ano
df_2009_2019.groupBy(F.year("pickup_datetime")).count().orderBy(F.year("pickup_datetime")).display()


Geração de dados histórico com 25 anos, com particionamento temporal devido a quantidade de registros

Com essa condição, trabalharemos com  1,611,611,035 de registros

In [0]:
from pyspark.sql import functions as F

# filtrando o histórico de 25 anos
df_historic = df_taxi.filter((F.col("pickup_datetime") >= "2000-01-01")) 

# adicionando colunas de partição temporal
df_historic = df_historic.withColumn("year", F.year("pickup_datetime"))
df_historic = df_historic.withColumn("month", F.month("pickup_datetime"))

print(f"Registros de 2000 a 2025 em diante: {df_historic.count():,}")      



In [0]:
catalog = 'nyc_transit'
schema = 'yellow_taxi'
table = 'trips_full_history'

In [0]:
import datetime

# grava o horario de inicio do processo
start_time = datetime.datetime.now()

# gravação da tabela histórica particionada
(df_historic.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .saveAsTable(f"{catalog}.{schema}.{table}"))

# mostra o tempo decorrido para execução do processo
print(f"Tempo decorrido: {datetime.datetime.now() - start_time}")
    

In [0]:
# análise do histórico da tabela gerada
display(df_historic.describe())

In [0]:
%sql

-- faz a contagem de registros por ano
SELECT year(pickup_datetime) as year, count(*) as count
FROM nyc_transit.yellow_taxi.trips_full_history
GROUP BY year(pickup_datetime)
ORDER BY year(pickup_datetime);



In [0]:
%sql

-- seleciona todos os campos da tabela yellow_taxi.trips_full_history onde o ano de pickup_datetime é 2008
SELECT *
FROM nyc_transit.yellow_taxi.trips_full_history
WHERE year(pickup_datetime) = 2008


In [0]:
%sql

-- agora vamos fazer o filtro incluindo a empresa que prestou o serviço
SELECT 
    year(pickup_datetime) as year, 
    vendor_id, -- Ajuda a ver qual empresa forneceu o dado
    count(*) as count
FROM nyc_transit.yellow_taxi.trips_full_history
WHERE pickup_datetime IS NOT NULL
GROUP BY 1, 2
ORDER BY 1;

In [0]:
%sql
-- Listar todas as tabelas relacionadas a taxi para ver se há divisões por ano/tipo
SHOW TABLES IN nyc_transit.yellow_taxi LIKE '*';

In [0]:
# lista as pastas principais de /databricks-datasets/nyctaxi
display(dbutils.fs.ls("/databricks-datasets/nyctaxi"))

In [0]:
# lista as pastas principais de /databricks-datasets/nyctaxi
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/reference"))

In [0]:
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata"))

In [0]:
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/fhv"))

In [0]:
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/fhvhv/"))

In [0]:
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/green/"))


In [0]:
# Definindo o caminho do dataset
path_fhv = "/databricks-datasets/nyctaxi/tripdata/fhv/*.csv.gz"

# Lendo os dados 
df_fhv = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", True)
          .load(path_fhv))

# Vamos ver o tamanho do "problema"
print(f"Total de registros: {df_fhv.count():,}")
# 1,611,611,035 registros (mais de 1bi)

# Mostrando o esquema do dataset
df_fhv.printSchema()

In [0]:
from pyspark.sql import functions as F

# Definindo o caminho do FHV
path_fhv = "/databricks-datasets/nyctaxi/tripdata/fhv/*.csv.gz"

# leitura dos dados FHV
df_fhv_raw = (spark.read
  .format("csv") 
  .option("header", True) 
  .load(path_fhv))

# padronização dos nomes dos campos
df_fhv_standard = df_fhv_raw.select(
    F.expr("try_to_timestamp(Pickup_DateTime)").alias("pickup_datetime"),
    F.expr("try_to_timestamp(DropOff_DateTime)").alias("dropoff_datetime"),
    F.lit("FHV").alias("taxi_type")
)

# Limpeza e filtros
df_fhv_filtered = df_fhv_standard.filter(F.col("pickup_datetime").isNotNull()) \
                                 .withColumn("year",  F.year("pickup_datetime")) \
                                 .withColumn("month", F.month("pickup_datetime")) \
                                 .filter("year >= 2000 AND year <= 2026")

# União com Yellow
df_yellow_standard = (spark.table(f"{catalog}.{schema}.trips_full_history")
                       .select("pickup_datetime", "dropoff_datetime")
                       .withColumn("taxi_type", F.lit("Yellow"))
                       .withColumn("year", F.year("pickup_datetime"))
                       .withColumn("month", F.month("pickup_datetime"))
                       )  

df_mobility_gold = df_yellow_standard.unionByName(df_fhv_filtered, allowMissingColumns=True)

# Escrevendo os dados no delta
(df_mobility_gold.write
  .format("delta")
  .mode("overwrite")
  .partitionBy("year")
  .saveAsTable(f"{catalog}.{schema}.gold_mobility_ny"))


In [0]:
%sql
SELECT year,
       taxi_type,
       count(*) as count
FROM nyc_transit.yellow_taxi.gold_mobility_ny
group by year, taxi_type
order by year, taxi_type

In [0]:
%sql
SELECT * FROM nyc_transit.yellow_taxi.gold_mobility_ny
WHERE year is null;

SELECT COUNT(*) FROM nyc_transit.yellow_taxi.gold_mobility_ny;

In [0]:
%sql
-- Execute em uma célula SQL antes de rodar o Python
DROP TABLE IF EXISTS nyc_transit.yellow_taxi.gold_mobility_ny;

In [0]:
from pyspark.sql import functions as F

# faz a mesma ingestão adicionando o caminho fhvhv

# lista de caminhos de todo o ecossistema FHV
paths = [
  "/databricks-datasets/nyctaxi/tripdata/fhv/*.csv.gz",
  "/databricks-datasets/nyctaxi/tripdata/fhvhv/*.csv.gz"
]

# leitura dos dados FHV
df_fhv_raw = (spark.read
  .format("csv") 
  .option("header", True) 
  .load(paths))

# normaliza os nomes das colunas em minúsculo
df_fhv_raw = df_fhv_raw.toDF(*[c.lower() for c in df_fhv_raw.columns])

# mapeamento dinâmico dos campos
cols = df_fhv_raw.columns

# Lógica para Pickup
if "pickup_datetime" in cols and "pickup_date" in cols:
    pickup_col = F.coalesce(F.expr("try_to_timestamp(pickup_datetime)"), F.expr("try_to_timestamp(pickup_date)"))
elif "pickup_datetime" in cols:
    pickup_col = F.expr("try_to_timestamp(pickup_datetime)")
else:
    pickup_col = F.expr("try_to_timestamp(pickup_date)")

# Lógica para Dropoff
if "dropoff_datetime" in cols and "dropoff_date" in cols:
    dropoff_col = F.coalesce(F.expr("try_to_timestamp(dropoff_datetime)"), F.expr("try_to_timestamp(dropoff_date)"))
elif "dropoff_datetime" in cols:
    dropoff_col = F.expr("try_to_timestamp(dropoff_datetime)")
else:
    dropoff_col = F.expr("try_to_timestamp(dropoff_date)")

# define dropoff_col como valor padrão em caso de pickup_col nulo
pickup_col = F.coalesce(pickup_col, dropoff_col)    

df_fhv_standard = df_fhv_raw.select(
    pickup_col.alias("pickup_datetime"),
    dropoff_col.alias("dropoff_datetime"),
    F.lit("FHV/Uber/Lyft").alias("taxi_type")
)

# Limpeza e filtros
df_fhv_filtered = df_fhv_standard.filter(F.col("pickup_datetime").isNotNull()) \
                                 .withColumn("year",  F.year("pickup_datetime")) \
                                 .withColumn("month", F.month("pickup_datetime")) \
                                 .filter("year >= 2000 AND year <= 2026")

# União com Yellow
df_yellow_standard = (spark.table(f"{catalog}.{schema}.trips_full_history")
                       .select("pickup_datetime", "dropoff_datetime")
                       .withColumn("taxi_type", F.lit("Yellow"))
                       .withColumn("year", F.year("pickup_datetime"))
                       .withColumn("month", F.month("pickup_datetime"))
                       )  

df_mobility_gold = df_yellow_standard.unionByName(df_fhv_filtered, allowMissingColumns=True)

# Escrevendo os dados no delta
(df_mobility_gold.write
  .format("delta")
  .mode("overwrite")
  .partitionBy("year")
  .saveAsTable(f"{catalog}.{schema}.gold_mobility_ny"))


In [0]:
%sql
SELECT COUNT(*) FROM nyc_transit.yellow_taxi.gold_mobility_ny;

SELECT COUNT(*) FROM nyc_transit.yellow_taxi.trips_full_history;

In [0]:
%sql
SELECT year,
       taxi_type,
       count(*) as count
FROM nyc_transit.yellow_taxi.gold_mobility_ny
group by year, taxi_type
order by year, taxi_type

In [0]:
from pyspark.sql import functions as F

# Definindo o caminho do FHV
path_fhv = "/databricks-datasets/nyctaxi/tripdata/fhv/*.csv.gz"

# leitura dos dados FHV
df_fhv_raw = (spark.read
  .format("csv") 
  .option("header", True) 
  .load(path_fhv))

# normaliza os nomes das colunas em minúsculo
df_fhv_raw = df_fhv_raw.toDF(*[c.lower() for c in df_fhv_raw.columns])

# mapeamento dinâmico dos campos
cols = df_fhv_raw.columns

# Lógica para Pickup
if "pickup_datetime" in cols and "pickup_date" in cols:
    pickup_col = F.coalesce(F.expr("try_to_timestamp(pickup_datetime)"), F.expr("try_to_timestamp(pickup_date)"))
elif "pickup_datetime" in cols:
    pickup_col = F.expr("try_to_timestamp(pickup_datetime)")
else:
    pickup_col = F.expr("try_to_timestamp(pickup_date)")

# Lógica para Dropoff
if "dropoff_datetime" in cols and "dropoff_date" in cols:
    dropoff_col = F.coalesce(F.expr("try_to_timestamp(dropoff_datetime)"), F.expr("try_to_timestamp(dropoff_date)"))
elif "dropoff_datetime" in cols:
    dropoff_col = F.expr("try_to_timestamp(dropoff_datetime)")
else:
    dropoff_col = F.expr("try_to_timestamp(dropoff_date)")

# define dropoff_col em caso de pickup_col nulo
pickup_col = F.coalesce(pickup_col, dropoff_col)

df_fhv_standard = df_fhv_raw.select(
    pickup_col.alias("pickup_datetime"),
    dropoff_col.alias("dropoff_datetime"),
    F.lit("FHV/Uber/Lyft").alias("taxi_type")
)

#df_fhv_datenull = df_fhv_standard.filter(F.col("pickup_datetime").isNull())   # 431,518,326 registros       

# Limpeza e filtros
df_fhv_filtered = df_fhv_standard.filter(F.col("pickup_datetime").isNotNull()) \
                                 .withColumn("year",  F.year("pickup_datetime")) \
                                 .withColumn("month", F.month("pickup_datetime")) \
                                 .filter("year >= 2000 AND year <= 2026")

df_fhv_dif2018 = df_fhv_standard.filter(F.col("pickup_datetime").isNotNull()) \
                                 .withColumn("year",  F.year("pickup_datetime")) \
                                 .withColumn("month", F.month("pickup_datetime")) \
                                 .filter("year <> 2018")

# mostra a quantidade de registros do dataframe
#print(f"Total de registros df_fhv_raw: {df_fhv_raw.count():,}")
print(f"Total de registros df_fhv_filtered: {df_fhv_filtered.count():,}")
#print(f"Total de registros df_fhv_datenull: {df_fhv_datenull.count():,}")
print(f"Total de registros df_fhv_dif2018: {df_fhv_dif2018.count():,}")

# lista os registros do df_fhv_filtered
#df_fhv_filtered.show()


In [0]:
from pyspark.sql import functions as F

# Definindo o caminho do FHV
path_fhv = "/databricks-datasets/nyctaxi/tripdata/fhvhv/*.csv.gz"

# leitura dos dados FHV
df_fhv_raw = (spark.read
  .format("csv") 
  .option("header", True) 
  .load(path_fhv))

#print(f"Total de registros df_fhv_raw: {df_fhv_raw.count():,}")
#df_fhv_raw.show()  

# normaliza os nomes das colunas em minúsculo
df_fhv_raw = df_fhv_raw.toDF(*[c.lower() for c in df_fhv_raw.columns])

# mapeamento dinâmico dos campos
cols = df_fhv_raw.columns

# Lógica para Pickup
if "pickup_datetime" in cols and "pickup_date" in cols:
    pickup_col = F.coalesce(F.expr("try_to_timestamp(pickup_datetime)"), F.expr("try_to_timestamp(pickup_date)"))
elif "pickup_datetime" in cols:
    pickup_col = F.expr("try_to_timestamp(pickup_datetime)")
else:
    pickup_col = F.expr("try_to_timestamp(pickup_date)")

# Lógica para Dropoff
if "dropoff_datetime" in cols and "dropoff_date" in cols:
    dropoff_col = F.coalesce(F.expr("try_to_timestamp(dropoff_datetime)"), F.expr("try_to_timestamp(dropoff_date)"))
elif "dropoff_datetime" in cols:
    dropoff_col = F.expr("try_to_timestamp(dropoff_datetime)")
else:
    dropoff_col = F.expr("try_to_timestamp(dropoff_date)")

# define dropoff_col em caso de pickup_col nulo
pickup_col = F.coalesce(pickup_col, dropoff_col)

df_fhv_standard = df_fhv_raw.select(
    pickup_col.alias("pickup_datetime"),
    dropoff_col.alias("dropoff_datetime"),
    F.lit("FHV/Uber/Lyft").alias("taxi_type")
)

#df_fhv_datenull = df_fhv_standard.filter(F.col("pickup_datetime").isNull())   # 431,518,326 registros       

# Limpeza e filtros
df_fhv_filtered = df_fhv_standard.filter(F.col("pickup_datetime").isNotNull()) \
                                 .withColumn("year",  F.year("pickup_datetime")) \
                                 .withColumn("month", F.month("pickup_datetime")) \
                                 .filter("year >= 2000 AND year <= 2026")

df_fhv_dif2019 = df_fhv_standard.filter(F.col("pickup_datetime").isNotNull()) \
                                 .withColumn("year",  F.year("pickup_datetime")) \
                                 .withColumn("month", F.month("pickup_datetime")) \
                                 .filter("year <> 2019")

# mostra a quantidade de registros do dataframe
#print(f"Total de registros df_fhv_raw: {df_fhv_raw.count():,}")
#print(f"Total de registros df_fhv_filtered: {df_fhv_filtered.count():,}")
#print(f"Total de registros df_fhv_datenull: {df_fhv_datenull.count():,}")
print(f"Total de registros df_fhv_dif2019: {df_fhv_dif2019.count():,}")

# lista os registros do df_fhv_filtered
df_fhv_dif2019.show()

In [0]:
catalog = 'nyc_transit'
schema = 'yellow_taxi'

**Eliminando outliers da tabela gold**

In [0]:
from pyspark.sql import functions as F

# 1. Carregando a tabela Gold
df_gold = spark.table(f"{catalog}.{schema}.gold_mobility_ny")

# 2. Definindo o limite superior (Mês atual)
# Usamos add_months para dar uma margem de segurança de 1 mês se desejar, 
# mas o ideal é o 'current_timestamp'
current_date_limit = F.current_timestamp()

# 3. Aplicando o filtro de Outliers Temporais
# Removemos datas futuras e datas extremamente antigas (lixo de sistema)
df_gold_clean = df_gold.filter(
    (F.col("pickup_datetime") <= current_date_limit) & 
    (F.col("pickup_datetime") >= "2009-01-01") # Focando na era digital confiável
)

# 4. Verificando quantos registros foram removidos
total_antes = df_gold.count()
total_depois = df_gold_clean.count()

print(f"Registros removidos (Outliers Futuros/Antigos): {total_antes - total_depois:,}")

In [0]:
import pandas as pd
from datetime import datetime

# Agregando com filtro de segurança
df_plot = (spark.table(f"{catalog}.{schema}.gold_mobility_ny")
           .filter(F.col("pickup_datetime") <= F.current_timestamp()) # Remove futuro
           .filter(F.col("pickup_datetime") >= "2009-01-01")          # Remove passado ruidoso
           .groupBy("year", "month", "taxi_type")
           .count()
           .orderBy("year", "month"))

pdf = df_plot.toPandas()

# Criando a data e removendo possíveis NaT (Not a Time)
pdf['date'] = pd.to_datetime(pdf[['year', 'month']].assign(day=1), errors='coerce')
pdf = pdf.dropna(subset=['date'])

# Filtro final no Pandas para garantir que não passamos de HOJE
pdf = pdf[pdf['date'] <= '2019-12-01'] #datetime.now()]

#display(pdf)

In [0]:
import plotly.express as px

color_map = {
    "Yellow": "#FFD700",
    "FHV/Uber/Lyft": "#222222"
}

fig = px.area(pdf, 
              x="date", 
              y="count", 
              color="taxi_type",
              title="Evolução da Mobilidade em NYC (2000-2026)",
              labels={"count": "Total de Viagens", "date": "Data", "taxi_type": "Categoria"},
              color_discrete_map=color_map,
              category_orders={"taxi_type": ["Yellow", "FHV/Uber/Lyft"]})

# ✅ Forçar fillcolor dos traces para bater com as cores definidas
for trace in fig.data:
    taxi_type = trace.name
    if taxi_type in color_map:
        hex_color = color_map[taxi_type]
        # Converter hex para rgba com opacidade (padrão do px.area é ~0.5)
        r = int(hex_color[1:3], 16)
        g = int(hex_color[3:5], 16)
        b = int(hex_color[5:7], 16)
        trace.fillcolor = f"rgba({r},{g},{b},0.5)"
        trace.line.color = hex_color

fig.update_layout(
    hovermode="x unified",
    plot_bgcolor="white",
    xaxis_title="Linha do Tempo",
    yaxis_title="Volume de Viagens (Mensal)",
    legend_title="Tipo de Serviço"
)

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey')

fig.show()

In [0]:
%sql
-- Query para sumário executivo
SELECT 
    taxi_type,
    MIN(year) as inicio_serie,
    MAX(year) as fim_serie,
    FORMAT_NUMBER(COUNT(*), 0) as total_viagens,
    ROUND(COUNT(*) / 1600000000 * 100, 2) as representatividade_pct
FROM nyc_transit.yellow_taxi.gold_mobility_ny
GROUP BY 1
ORDER BY 4 DESC;

In [0]:
%sql
WITH mensal_agrupado AS (
    SELECT 
        year, 
        month, 
        taxi_type, 
        COUNT(*) as total_viagens
    FROM nyc_transit.yellow_taxi.gold_mobility_ny
    WHERE year BETWEEN 2009 AND 2019
    GROUP BY 1, 2, 3
),
total_mes AS (
    SELECT 
        year, 
        month, 
        SUM(total_viagens) as volume_total_mercado
    FROM mensal_agrupado
    GROUP BY 1, 2
)
SELECT 
    m.year, 
    m.month, 
    m.taxi_type,
    m.total_viagens,
    ROUND((m.total_viagens / t.volume_total_mercado) * 100, 2) as market_share_pct
FROM mensal_agrupado m
JOIN total_mes t ON m.year = t.year AND m.month = t.month
ORDER BY m.year, m.month, m.taxi_type;

In [0]:
%sql
-- executa a otimização na tabela gold
OPTIMIZE nyc_transit.yellow_taxi.gold_mobility_ny ZORDER BY (taxi_type) ;


In [0]:
%sql
-- elimina tabelas que não são mais usadas
drop table if exists nyc_transit.yellow_taxi.trip_optimized;
drop table if exists nyc_transit.yellow_taxi.trips_full_history;

In [0]:
%sql

select * from nyc_transit.yellow_taxi.gold_mobility_ny; where dropoff_datetime is not null;

select * from nyc_transit.yellow_taxi.gold_market_share_summary;
